In [1]:
import numpy as np
import pandas as pd
from sklearn import linear_model

In [2]:
df = pd.read_csv('/content/drive/MyDrive/Linear Regression/loan_approval_dataset.csv')
df

,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected
...,...,...,...,...,...,...,...,...,...,...,...,...
4264,5,Graduate,Yes,1000000,2300000,12,317,2800000,500000,3300000,800000,Rejected
4265,0,Not Graduate,Yes,3300000,11300000,20,559,4200000,2900000,11000000,1900000,Approved
4266,2,Not Graduate,No,6500000,23900000,18,457,1200000,12400000,18100000,7300000,Rejected
4267,1,Not Graduate,No,4100000,12800000,8,780,8200000,700000,14100000,5800000,Approved


In [3]:
df.columns = df.columns.str.strip()

df['education'] = df['education'].str.strip()
df['self_employed'] = df['self_employed'].str.strip()
df['loan_status'] = df['loan_status'].str.strip()

In [4]:
print(df['education'].unique())
print(df['self_employed'].unique())
print(df['loan_status'].unique())

['Graduate' 'Not Graduate']
['No' 'Yes']
['Approved' 'Rejected']


In [5]:
from sklearn.preprocessing import LabelEncoder

le_education = LabelEncoder()
le_self_employed = LabelEncoder()
le_loan_status = LabelEncoder()

df['education'] = le_education.fit_transform(df['education'])
df['self_employed'] = le_self_employed.fit_transform(df['self_employed'])
df['loan_status'] = le_loan_status.fit_transform(df['loan_status'])

In [6]:
print(le_education.classes_)
print(le_self_employed.classes_)
print(le_loan_status.classes_)

['Graduate' 'Not Graduate']
['No' 'Yes']
['Approved' 'Rejected']


In [7]:
from sklearn.model_selection import train_test_split
X = df[['no_of_dependents',
        'education',
        'self_employed',
        'income_annum',
        'loan_amount',
        'loan_term',
        'cibil_score',
        'residential_assets_value',
        'commercial_assets_value',
        'luxury_assets_value',
        'bank_asset_value']]

y = df['loan_status']

In [8]:
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
lor = linear_model.LogisticRegression(max_iter=1000)
lor.fit(x_train, y_train)

LogisticRegression(max_iter=1000)

In [10]:
lor.score(x_train, y_train)

0.7947291361639824

In [11]:
lor.score(x_test, y_test)

0.7985948477751756

In [12]:
prediction = lor.predict([[
    2,          # no_of_dependents
    0,          # education
    0,          # self_employed
    5000000,    # income_annum
    10000000,   # loan_amount
    10,         # loan_term
    750,        # cibil_score
    3000000,    # residential_assets_value
    2000000,    # commercial_assets_value
    5000000,    # luxury_assets_value
    1500000     # bank_asset_value
]])

result = le_loan_status.inverse_transform(prediction)

print("Loan Status:", result[0])

Loan Status: Approved


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [13]:
training_accuracy = lor.score(x_train, y_train)
testing_accuracy = lor.score(x_test, y_test)

print("Training Accuracy:", training_accuracy * 100, "%")
print("Testing Accuracy:", testing_accuracy * 100, "%")

Training Accuracy: 79.47291361639824 %
Testing Accuracy: 79.85948477751757 %


In [14]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = lor.predict(x_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[488  48]
 [124 194]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.91      0.85       536
           1       0.80      0.61      0.69       318

    accuracy                           0.80       854
   macro avg       0.80      0.76      0.77       854
weighted avg       0.80      0.80      0.79       854



In [15]:
import joblib

joblib.dump(lor, "loan_approval_model.pkl")
joblib.dump(le_education, "education_encoder.pkl")
joblib.dump(le_self_employed, "self_employed_encoder.pkl")
joblib.dump(le_loan_status, "loan_status_encoder.pkl")

['loan_status_encoder.pkl']

In [16]:
import os
print(os.listdir('/content'))

['.config', 'self_employed_encoder.pkl', 'education_encoder.pkl', 'requirements.txt', 'drive', 'app.py', 'loan_approval_model.pkl', 'loan_status_encoder.pkl', 'sample_data']


In [17]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib

model = joblib.load("loan_approval_model.pkl")
education_encoder = joblib.load("education_encoder.pkl")
self_employed_encoder = joblib.load("self_employed_encoder.pkl")
loan_status_encoder = joblib.load("loan_status_encoder.pkl")

st.set_page_config(
    page_title="Loan Approval Predictor",
    page_icon="🏦",
    layout="centered"
)

st.title("🏦 Loan Approval Prediction System")

st.write(
    "Enter the applicant's details to predict whether the loan "
    "is likely to be approved or rejected."
)

with st.form("loan_form"):

    no_of_dependents = st.number_input(
        "Number of Dependents",
        min_value=0,
        max_value=10,
        value=0,
        step=1
    )

    education = st.selectbox(
        "Education",
        education_encoder.classes_.tolist()
    )

    self_employed = st.selectbox(
        "Self Employed",
        self_employed_encoder.classes_.tolist()
    )

    income_annum = st.number_input(
        "Annual Income",
        min_value=0,
        value=5000000,
        step=100000
    )

    loan_amount = st.number_input(
        "Loan Amount",
        min_value=0,
        value=10000000,
        step=100000
    )

    loan_term = st.number_input(
        "Loan Term",
        min_value=1,
        max_value=30,
        value=10,
        step=1
    )

    cibil_score = st.number_input(
        "CIBIL Score",
        min_value=300,
        max_value=900,
        value=700,
        step=1
    )

    residential_assets_value = st.number_input(
        "Residential Assets Value",
        min_value=0,
        value=3000000,
        step=100000
    )

    commercial_assets_value = st.number_input(
        "Commercial Assets Value",
        min_value=0,
        value=2000000,
        step=100000
    )

    luxury_assets_value = st.number_input(
        "Luxury Assets Value",
        min_value=0,
        value=5000000,
        step=100000
    )

    bank_asset_value = st.number_input(
        "Bank Asset Value",
        min_value=0,
        value=1500000,
        step=100000
    )

    submitted = st.form_submit_button("Predict Loan Status")

if submitted:

    education_encoded = education_encoder.transform([education])[0]
    self_employed_encoded = self_employed_encoder.transform(
        [self_employed]
    )[0]

    input_data = pd.DataFrame([{
        "no_of_dependents": no_of_dependents,
        "education": education_encoded,
        "self_employed": self_employed_encoded,
        "income_annum": income_annum,
        "loan_amount": loan_amount,
        "loan_term": loan_term,
        "cibil_score": cibil_score,
        "residential_assets_value": residential_assets_value,
        "commercial_assets_value": commercial_assets_value,
        "luxury_assets_value": luxury_assets_value,
        "bank_asset_value": bank_asset_value
    }])

    prediction = model.predict(input_data)

    result = loan_status_encoder.inverse_transform(prediction)[0].strip()

    if result.lower() == "approved":
        st.success(f"Loan Status: {result}")
    else:
        st.error(f"Loan Status: {result}")

Overwriting app.py


In [18]:
%%writefile requirements.txt
streamlit
pandas
scikit-learn
joblib

Overwriting requirements.txt


In [19]:
from google.colab import files

files.download('app.py')
files.download('requirements.txt')
files.download('loan_approval_model.pkl')
files.download('education_encoder.pkl')
files.download('self_employed_encoder.pkl')
files.download('loan_status_encoder.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
print(le_loan_status.classes_)

['Approved' 'Rejected']


In [21]:
print(le_education.classes_)
print(le_self_employed.classes_)
print(le_loan_status.classes_)

['Graduate' 'Not Graduate']
['No' 'Yes']
['Approved' 'Rejected']
